# 6j — Fitting Gamma & Weibull to the weighted degree distribution

Source spec: `inst/7_fitting_dist.md`.

We model the **duration-weighted degree** — each participant-day's contacts summed
with a per-contact weight derived from contact duration:

- `>4h` contacts are the **reference** (weight = 1; `>4h` treated as 4h),
- shorter contacts are down-weighted by their duration **midpoint**,
- missing (NA) durations are treated as `<5min` (2.5 min, level 1).

This equals `duration_weight(d, d_max=240)` (`src/degree_dist.jl`). The weighted degree
is **continuous**, so we fit the continuous **Gamma** and **Weibull** distributions
(MLE), separately for **home** and **non-home**, and — unlike earlier notebooks that
drop zeros — also **including zero-contact** participant-days (via a small-value shift).

In [ ]:
include("main_utils.jl")
include("data_setup.jl")
include("comix_uk_time_series.jl")
include("vis_utils.jl")

using Distributions

default_plot_setting()

## §1 Data reading

Same loader and Jul-2021–Mar-2022 window as `5j_group_contacts.ipynb`.

In [ ]:
df, df_part = read_comix_uk_contact_raw()
df_part = @subset(df_part, Date(2021, 7, 1) .<= :date .< Date(2022, 4, 1))
df = innerjoin(df, @select(df_part, :part_id_d, :date),
               on = [:part_id_d, :date])

println("# contacts (after Jul21–Mar22 filter): ", nrow(df))
println("# participant-diary-days:             ",
        nrow(unique(@select(df_part, :part_id_d, :date))))

## §2 Weighted degree (home & non-home)

`d_max = 240` min so that `>4h ⇒ weight 1`, `1–4h ⇒ 150/240 = 0.625`, …,
`<5min ⇒ 2.5/240 ≈ 0.0104`. NA durations fall to level 1 (`<5min`) inside
`duration_weight`. Participant-days with no contacts in a setting get `0.0`.

In [ ]:
const DMAX = 240

# sanity: the weighting scheme behaves as specified
@assert duration_weight(5, DMAX) == 1.0          # >4h is the reference
@assert duration_weight(4, DMAX) == 150 / DMAX   # 1–4h
@assert duration_weight(1, DMAX) == 2.5 / DMAX   # <5min (and NA)

wdeg = Dict(
    :home    => contact_degrees(df, df_part; setting = :home,    weighted = true, d_max = DMAX),
    :nonhome => contact_degrees(df, df_part; setting = :nonhome, weighted = true, d_max = DMAX),
)

for s in (:home, :nonhome)
    w = wdeg[s]
    println(rpad(string(s), 9), " | n = ", length(w),
            " | zeros = ", count(==(0.0), w),
            " (", round(100 * count(==(0.0), w) / length(w); digits = 1), "%)")
end

## §3 Build the fit datasets per setting

The continuous fits have support `> 0`, so zeros need handling. We build **three**
zero-treatment variants per setting:

- **with zeros** — replace each zero with a tiny `EPS` (the smallest single-contact
  weight, `2.5/240 ≈ 0.0104`) so no-contact participant-days enter the fit;
- **without zeros** — positives only (the conventional zero-excluded view);
- **without zeros (<0.1→0)** — treat any weighted degree `< 0.1` as a zero and drop it,
  then fit the remainder. This removes participant-days whose entire contact budget is
  just a few very short (`<5min`) contacts (a single `<5min` contact weighs `≈0.0104`).

In [ ]:
const EPS = duration_weight(1, DMAX)   # ≈ 0.0104, smallest single-contact weight
const WDEG_ZERO_THRESH = 0.1           # weighted degree below this is treated as zero

shift_zeros(w)   = [x > 0 ? x : EPS for x in w]        # "with zeros"
positives(w)     = filter(>(0), w)                     # "without zeros"
threshold_pos(w) = filter(>=(WDEG_ZERO_THRESH), w)     # "<0.1 → zero", then drop zeros

# summary of the positive weighted degree per setting
summ = DataFrame(setting = String[], n_pos = Int[], n_ge_0p1 = Int[], mean = Float64[],
                 median = Float64[], q95 = Float64[], max = Float64[])
for s in (:home, :nonhome)
    p = positives(wdeg[s])
    push!(summ, (string(s), length(p), length(threshold_pos(wdeg[s])),
                 mean(p), median(p), quantile(p, 0.95), maximum(p)))
end
summ

## §4 Fit the candidate distributions (MLE)

Four candidates, fit by maximum likelihood to every (setting × zero-variant):

| distribution | params (`param1`, `param2`) | fit |
|---|---|---|
| **Gamma** | shape α, scale θ | `fit_mle` |
| **Weibull** | shape α, scale θ | `fit_mle` (MoM-initialised) |
| **Lomax** (Pareto-II, heavy tail) | shape α, scale θ | profile-likelihood MLE (below) |
| **LogNormal** | log-mean μ, log-sd σ | `fit_mle` |

`Lomax` has no `fit_mle` in Distributions.jl, so we maximise its likelihood directly:
for fixed scale θ the shape has the closed form `α̂(θ) = n / Σ log(1 + xᵢ/θ)`, leaving a
1-D profile in θ that we optimise by golden-section search. (For light-tailed data the
optimum runs to large θ — the exponential limit of the Lomax.)

We report each fit's two parameters, log-likelihood, and AIC (`2k − 2·loglik`, `k = 2`).

In [ ]:
function fit_weibull(x)
    # MoM init: for Weibull, std(log x) = π/(√6·k). Newton from the default
    # alpha0 = 1 diverges on the zero-spiked "with zeros" data; this init is robust.
    k0 = pi / (sqrt(6) * std(log.(x)))
    return fit_mle(Weibull, x; alpha0 = k0)
end

# Lomax MLE via profile likelihood: α̂(θ) = n / Σ log(1+xᵢ/θ); 1-D golden-section in θ.
# `Lomax(α, θ)` is the project distribution from src/distributions/poisson_mixture.jl.
function fit_lomax(x)
    n = length(x)
    S(θ) = sum(log1p.(x ./ θ))
    negll(logθ) = (θ = exp(logθ); s = S(θ); α = n / s; -(n * log(α) - n * log(θ) - (α + 1) * s))
    φ = (sqrt(5) - 1) / 2
    a, b = log(1e-3), log(1e6)
    c = b - φ * (b - a); d = a + φ * (b - a)
    fc = negll(c); fd = negll(d)
    for _ in 1:300
        if fc < fd
            b, d, fd = d, c, fc; c = b - φ * (b - a); fc = negll(c)
        else
            a, c, fc = c, d, fd; d = a + φ * (b - a); fd = negll(d)
        end
        (b - a) < 1e-9 && break
    end
    θ = exp((a + b) / 2)
    return Lomax(n / S(θ), θ)
end

fit_all(x) = (gamma     = fit_mle(Gamma, x),
              weibull   = fit_weibull(x),
              lomax     = fit_lomax(x),
              lognormal = fit_mle(LogNormal, x))

dataset(s, v) = v === :with_zeros  ? shift_zeros(wdeg[s])   :
                v === :thresholded ? threshold_pos(wdeg[s]) :
                                     positives(wdeg[s])

settings = (:home, :nonhome)
variants = (:with_zeros, :without_zeros, :thresholded)
DIST_KEYS = ((:gamma, "Gamma"), (:weibull, "Weibull"), (:lomax, "Lomax"), (:lognormal, "LogNormal"))

data = Dict((s, v) => dataset(s, v) for s in settings, v in variants)
fits = Dict((s, v) => fit_all(data[(s, v)]) for s in settings, v in variants)

aic(d, x) = 2 * 2 - 2 * sum(logpdf.(d, x))

# (param1, param2) per distribution — see §4 table for their meaning.
dist_params(d::Gamma)     = (Distributions.shape(d), Distributions.scale(d))
dist_params(d::Weibull)   = (Distributions.shape(d), Distributions.scale(d))
dist_params(d::Lomax)     = (d.α, d.θ)
dist_params(d::LogNormal) = (d.μ, d.σ)

restab = DataFrame(setting = String[], zeros = String[], dist = String[],
                   param1 = Float64[], param2 = Float64[],
                   loglik = Float64[], AIC = Float64[])
for s in settings, v in variants
    x = data[(s, v)]
    for (k, name) in DIST_KEYS
        d = getfield(fits[(s, v)], k)
        p1, p2 = dist_params(d)
        push!(restab, (string(s), string(v), name, p1, p2, sum(logpdf.(d, x)), aic(d, x)))
    end
end
restab

## §5 Visualisation — pdf & CCDF

One figure per setting; rows are the three zero-variants (*with zeros*, *without zeros*,
*without zeros (<0.1→0)*) and columns are the empirical **pdf** (density histogram,
semilog-y) and **CCDF** (survival, log–log). Each panel overlays the four fitted curves
(Gamma, Weibull, Lomax, LogNormal) on the empirical series the fit was trained on. The
lower y-limit is set one decade below the lowest observed value.

In [ ]:
const BINWIDTH = 0.5
const YLIM_PAD = 10.0   # leave one log10 decade below the lowest observed value (room for a ytick)

const DIST_STYLE = ((:gamma, "Gamma", :red), (:weibull, "Weibull", :blue),
                    (:lomax, "Lomax", :green), (:lognormal, "LogNormal", :purple))
const VLAB = Dict(:with_zeros => "with zeros", :without_zeros => "without zeros",
                  :thresholded => "without zeros (<0.1→0)")

# Lowest observed density in the pdf histogram = density of the least-populated
# (non-empty) bin. Mirrors the edges used by `plot_pdf_hist!` (0:bw:max+bw).
function obs_pdf_floor(x, bw)
    nb = ceil(Int, maximum(x) / bw) + 1
    counts = zeros(Int, nb)
    for v in x
        counts[clamp(floor(Int, v / bw) + 1, 1, nb)] += 1
    end
    dens = counts ./ (length(x) * bw)
    return minimum(filter(>(0), dens))
end

function pdf_panel(x, ft; title)
    p = plot_pdf_hist!(plot(; title = title), x; label = "empirical", binwidth = BINWIDTH)
    xs = range(EPS, maximum(x); length = 400)
    for (k, name, col) in DIST_STYLE
        plot!(p, xs, pdf.(getfield(ft, k), xs); label = name, color = col, lw = 2)
    end
    ylims!(p, (obs_pdf_floor(x, BINWIDTH) / YLIM_PAD, ylims(p)[2]))   # one decade below lowest observed density
    return p
end

function ccdf_panel(x, ft; title)
    p = plot_ccdf_continuous!(plot(; title = title, xaxis = :log10), x; label = "empirical")
    xs = sort(filter(>(0), x))
    for (k, name, col) in DIST_STYLE
        plot!(p, xs, ccdf.(getfield(ft, k), xs); label = name, color = col, lw = 2)
    end
    ylims!(p, (1 / (length(xs) * YLIM_PAD), 1.0))   # one decade below lowest observed CCDF (1/n)
    return p
end

# one figure per setting: rows = zero-variants, cols = {pdf, CCDF}
function setting_figure(s)
    panels = Plots.Plot[]
    for v in variants
        x  = data[(s, v)]
        ft = fits[(s, v)]
        push!(panels, pdf_panel(x, ft;  title = "$s — pdf ($(VLAB[v]))"))
        push!(panels, ccdf_panel(x, ft; title = "$s — CCDF ($(VLAB[v]))"))
    end
    plot(panels...; layout = (length(variants), 2), size = (1000, 360 * length(variants)))
end

In [ ]:
setting_figure(:home)

In [ ]:
setting_figure(:nonhome)

## §6 Takeaways

Best fit by AIC (lowest, compared *within* each dataset — the three zero-variants are
different datasets and not cross-comparable):

| setting | with zeros | without zeros | without zeros (<0.1→0) |
|---|---|---|---|
| **home** | Gamma | Weibull | Gamma |
| **non-home** | **Lomax** | Weibull | LogNormal |

- **Home is light-tailed**: Gamma/Weibull win, and the heavy-tailed **Lomax** runs to
  its exponential limit (θ→large) — it has no heavy tail to exploit.
- **Non-home is heavier-tailed**: with the full zero-spike included (*with zeros*),
  **Lomax** wins decisively; once the tiny-weight participant-days are removed
  (*without zeros* / *<0.1→0*) the bulk is better described by Weibull / LogNormal.
- The **`<0.1→0`** variant strips participant-days whose whole contact budget is a
  handful of `<5min` contacts (weighted degree `< 0.1`). It removes the near-`EPS`
  spike that dominates the *with zeros* fits, so shape parameters rise back above 1 and
  the fitted bulk re-centres on genuine multi-contact days.
- Non-home still has the longest empirical tail (weighted degree up to ≈94); Lomax
  tracks it best, the light-tailed Gamma/Weibull/LogNormal under-capture it.

See `restab` for the full parameter / log-likelihood / AIC table.

## §7 Minimum group contact size

The participant file records group-contact sizes in the nine `multiple_contacts_*`
columns (`_MC_COLS_UK`: {child, adult, older-adult} × {work, school, other}). Raw
values are a mix of integers (the reported group size), `"yes"`/`"no"`, and free-text.
Parsing the numeric entries across all nine columns — mirroring the `:has_group_num`
branch of 5j's `classify_mc` (strictly positive integer/float) — gives the distribution
of reported group sizes; its minimum is the smallest reported group contact size.

In [ ]:
# Parse a multiple_contacts_* value into a positive group size (else `nothing`),
# mirroring the `:has_group_num` branch of 5j's classify_mc.
function mc_size(v)
    (ismissing(v) || v == "NA" || v == "") && return nothing
    n = tryparse(Int, v);     n !== nothing && return n > 0 ? float(n) : nothing
    f = tryparse(Float64, v); f !== nothing && return f > 0 ? f : nothing
    return nothing
end

group_sizes = Float64[]
for c in _MC_COLS_UK, v in df_part[!, c]
    s = mc_size(v)
    s !== nothing && push!(group_sizes, s)
end

println("# numeric group-size entries (across all 9 columns): ", length(group_sizes))
println("minimum group contact size: ", Int(minimum(group_sizes)))
println("maximum group contact size: ", Int(maximum(group_sizes)))

# distribution of the smallest reported sizes
size_tab = @chain DataFrame(size = Int.(group_sizes)) begin
    groupby(:size)
    combine(nrow => :count)
    sort(:size)
end
first(size_tab, 10)

## §8 Age-pair degree distributions (CIS bins)

Stratify the **duration-weighted** degree (`d_max = DMAX = 240`, as above) by the
**age pair** *(participant CIS bin → contactee CIS bin)*, separately for **home**
and **non-home**, then fit a **Weibull** to each cell's positive degrees
(*without zeros*).

**CIS age bins** (7): `2–10, 11–15, 16–24, 25–34, 35–49, 50–69, 70+`.

Both the participant age (`part_age` group string) and the contactee age
(`[cnt_age_est_min, cnt_age_est_max]`) are reported as **intervals** that do not
line up with the CIS bins. Each age is assigned to a CIS bin as follows: among the
CIS bins that **overlap** the reported interval, pick one by sampling
**proportional to the bin's population size** (`populations.csv`, `age_school`,
England). An interval contained in a single bin is assigned deterministically;
"NA"/"Don't know" (→ `[0,120]`) draws across all 7 bins by population. A single
seeded draw (`MersenneTwister(1236)`) keeps it reproducible.

Note: `read_comix_uk_contact_raw()` is *not* adult-filtered, so child participant
bins (2–10, 11–15) populate too.

In [ ]:
using CSV, Random, StatsBase

# CIS bins + population weights (England, `age_school` rows of populations.csv).
pop_df  = CSV.read("../inc2prev/data-processed/populations.csv", DataFrame)
pop_age = @subset(pop_df, :level .== "age_school", :geography .== "England")
_asint(x)   = x isa AbstractString ? parse(Int, x)     : Int(x)
_asfloat(x) = x isa AbstractString ? parse(Float64, x) : Float64(x)
pop_age = @transform(pop_age, :lo = _asint.(:lower_age_limit), :pop = _asfloat.(:population))
sort!(pop_age, :lo)

CIS_LO  = pop_age.lo                       # [2,11,16,25,35,50,70]
CIS_HI  = vcat(CIS_LO[2:end] .- 1, 120)    # [10,15,24,34,49,69,120]
CIS_POP = pop_age.pop
N_BIN   = length(CIS_LO)
CIS_LAB = [CIS_LO[j] == CIS_LO[end] ? "$(CIS_LO[j])+" : "$(CIS_LO[j])-$(CIS_HI[j])" for j in 1:N_BIN]

# ---- interval parsing & population-weighted bin assignment ----
_toint(s) = (ismissing(s) || s == "NA") ? nothing : tryparse(Int, String(s))

"Parse a participant age-group string \"lo-hi\" → (lo,hi); NA/unparseable → (0,120)."
function parse_age_interval(s)
    (ismissing(s) || s == "NA") && return (0, 120)
    parts = split(String(s), "-")
    length(parts) == 2 || return (0, 120)
    a = tryparse(Int, parts[1]); b = tryparse(Int, parts[2])
    (a === nothing || b === nothing) ? (0, 120) : (a, b)
end

"Contactee interval from (est_min, est_max); NA/unparseable → (0,120)."
function interval_from_minmax(mn, mx)
    a = _toint(mn); b = _toint(mx)
    (a === nothing || b === nothing) ? (0, 120) : (a, b)
end

overlapping(a, b) = [j for j in 1:N_BIN if a <= CIS_HI[j] && b >= CIS_LO[j]]

"CIS bin for interval [a,b]: deterministic if one overlapping bin, else sampled ∝ population."
function assign_bin(a, b, rng)
    cand = overlapping(a, b)
    isempty(cand)        && return argmin(abs.(CIS_LO .- a))   # below youngest bin: nearest
    length(cand) == 1    && return cand[1]
    return sample(rng, cand, Weights(CIS_POP[cand]))
end

is_ambiguous(a, b) = length(overlapping(a, b)) > 1

DataFrame(bin = CIS_LAB, lo = CIS_LO, hi = CIS_HI, population_M = round.(CIS_POP ./ 1e6; digits = 2))

In [ ]:
# Build a contact table with the contactee age columns (row-aligned in the arrow),
# deriving duration_multi / cnt_home with the loader's own helpers, then restrict to
# the same participant-days as §1. (We re-read rather than join: the `contact` index
# is not a unique per-contact key, so a key-join would explode rows.)
craw = read_arrow_df("../dt_comix_no_public/contacts_uk.arrow";
    cols = [:part_wave_uid, :date, :cnt_home, :cnt_minutes_max, :cnt_total_time,
            :cnt_age_est_min, :cnt_age_est_max])
craw = @select(craw,
    :part_id_d      = :part_wave_uid,
    :date,
    :cnt_home,
    :duration_multi = _uk_duration_multi.(:cnt_minutes_max, :cnt_total_time),
    :cnt_age_est_min, :cnt_age_est_max)
standardise_cnt_home_values!(craw)
craw = @subset(craw, Date(2021, 7, 1) .<= :date .< Date(2022, 4, 1))

rng = MersenneTwister(1236)

# participant CIS bin: one draw per participant-day, then join onto contacts
piv = parse_age_interval.(df_part.part_age)
df_part[!, :part_bin] = [assign_bin(a, b, rng) for (a, b) in piv]
dfA = innerjoin(craw, unique(@select(df_part, :part_id_d, :date, :part_bin)),
                on = [:part_id_d, :date])

# contactee CIS bin: one draw per contact
civ = [interval_from_minmax(mn, mx) for (mn, mx) in zip(dfA.cnt_age_est_min, dfA.cnt_age_est_max)]
dfA[!, :cnt_bin] = [assign_bin(a, b, rng) for (a, b) in civ]

@assert nrow(dfA) == nrow(df)   # same contact population as §1
println("contacts = ", nrow(dfA))
println("contactee ages ambiguous (span >1 CIS bin): ",
        round(100 * mean(is_ambiguous.(first.(civ), last.(civ))); digits = 1), "%")
println("participant-day CIS-bin counts: ",
        [(CIS_LAB[i], count(==(i), df_part.part_bin)) for i in 1:N_BIN])
DataFrame(part_bin = CIS_LAB,
          n_contacts_as_participant = [count(==(i), dfA.part_bin) for i in 1:N_BIN])

In [ ]:
# Per (participant bin i, contactee bin j) and setting: the positive weighted degrees
# (one per participant-day with ≥1 such contact → strictly > 0, i.e. "without zeros").
function cell_wdeg(setting)
    sub = setting === :home ? @subset(dfA, :cnt_home .== "true") :
                              @subset(dfA, :cnt_home .== "false")
    sub = @transform(sub, :w = duration_weight.(:duration_multi, DMAX))
    g = combine(groupby(sub, [:part_id_d, :date, :part_bin, :cnt_bin]), :w => sum => :deg)
    D = Dict{Tuple{Int,Int},Vector{Float64}}()
    for r in eachrow(g)
        push!(get!(D, (Int(r.part_bin), Int(r.cnt_bin)), Float64[]), r.deg)
    end
    return D
end
cellw = Dict(:home => cell_wdeg(:home), :nonhome => cell_wdeg(:nonhome))

MIN_N = 30   # skip cells with fewer positive participant-days
weib  = Dict{Symbol,Dict{Tuple{Int,Int},Weibull}}(:home => Dict(), :nonhome => Dict())

agetab = DataFrame(setting = String[], part_bin = String[], cnt_bin = String[],
                   n = Int[], shape = Float64[], scale = Float64[], loglik = Float64[])
for s in (:home, :nonhome), i in 1:N_BIN, j in 1:N_BIN
    x = get(cellw[s], (i, j), Float64[])
    if length(x) >= MIN_N
        d = fit_weibull(x)                      # reuse §4 MoM-initialised fit_weibull
        weib[s][(i, j)] = d
        push!(agetab, (string(s), CIS_LAB[i], CIS_LAB[j], length(x),
                       Distributions.shape(d), Distributions.scale(d), sum(logpdf.(d, x))))
    else
        push!(agetab, (string(s), CIS_LAB[i], CIS_LAB[j], length(x), NaN, NaN, NaN))
    end
end
println("cells fitted: ", sum(.!isnan.(agetab.shape)), " / ", nrow(agetab))
agetab

In [ ]:
# 7×7 grid plotters: empirical (black) overlaid with the fitted Weibull (red).
# Panel label is annotated in the upper-right corner (not a title) to free up
# plotting area; axis limits are shared across all panels for comparability.
mkpath("../res")
emp_ccdf(x) = (xs = sort(x); (xs, (length(xs):-1:1) ./ length(xs)))
# empirical pdf as density points at bin centres (plotted as markers — a filled
# histogram on a log y-axis renders as broken slivers when saved to png)
function emp_pdf_pts(x; bw = 0.5)
    edges = 0:bw:(maximum(x) + bw)
    h = StatsBase.fit(Histogram, x, edges)
    dens = h.weights ./ (sum(h.weights) * bw)
    centers = (edges[1:end-1] .+ edges[2:end]) ./ 2
    (centers, dens)
end

# --- shared limits over every populated cell (both settings) ---
_allpos = [v for s in (:home, :nonhome) for v in values(cellw[s]) if length(v) >= MIN_N]
_xmin = minimum(minimum.(_allpos))
_xmax = maximum(maximum.(_allpos))
_maxn = maximum(length.(_allpos))

CCDF_XLIM = (10.0^floor(log10(_xmin)), 10.0^ceil(log10(_xmax)))   # log-log
CCDF_YLIM = (10.0^floor(log10(1 / _maxn)), 1.3)

function _pdf_density_range()
    lo = Inf; hi = -Inf
    for x in _allpos
        _, dens = emp_pdf_pts(x)
        pos = filter(>(0), dens)
        isempty(pos) || (lo = min(lo, minimum(pos)); hi = max(hi, maximum(pos)))
    end
    (lo, hi)
end
_dlo, _dhi = _pdf_density_range()
PDF_XLIM = (0.0, _xmax * 1.02)
PDF_YLIM = (10.0^floor(log10(_dlo)), 10.0^ceil(log10(_dhi)))

# place the panel label in the upper-right corner, respecting log/linear axes
function _corner!(p, label, xlim, ylim; logx, logy)
    fx(t) = logx ? 10.0^(log10(xlim[1]) + t * (log10(xlim[2]) - log10(xlim[1]))) :
                   xlim[1] + t * (xlim[2] - xlim[1])
    fy(t) = logy ? 10.0^(log10(ylim[1]) + t * (log10(ylim[2]) - log10(ylim[1]))) :
                   ylim[1] + t * (ylim[2] - ylim[1])
    annotate!(p, fx(0.97), fy(0.95), text(label, 5, :right, :top))
    p
end

function age_ccdf_panel(s, i, j)
    x = get(cellw[s], (i, j), Float64[])
    p = plot(; legend = false, tickfontsize = 4, guidefontsize = 5,
             xscale = :log10, yscale = :log10, xlim = CCDF_XLIM, ylim = CCDF_YLIM)
    if length(x) >= MIN_N
        xs, cc = emp_ccdf(x)
        plot!(p, xs, cc; seriestype = :steppost, color = :black, lw = 0.7)
        gx = exp10.(range(log10(minimum(xs)), log10(maximum(xs)); length = 150))
        plot!(p, gx, ccdf.(weib[s][(i, j)], gx); color = :red, lw = 1.2)
    end
    _corner!(p, "$(CIS_LAB[i])→$(CIS_LAB[j])\nn=$(length(x))", CCDF_XLIM, CCDF_YLIM;
             logx = true, logy = true)
end

function age_pdf_panel(s, i, j)
    x = get(cellw[s], (i, j), Float64[])
    p = plot(; legend = false, tickfontsize = 4, guidefontsize = 5,
             yscale = :log10, xlim = PDF_XLIM, ylim = PDF_YLIM)
    if length(x) >= MIN_N
        histogram!(p, x; bins = 0:0.5:(maximum(x) + 0.5), normalize = :pdf,
                   color = :gray, alpha = 0.4, linealpha = 0.3)
        gx = range(1e-3, maximum(x); length = 150)
        plot!(p, gx, pdf.(weib[s][(i, j)], gx); color = :red, lw = 1.2)
    end
    _corner!(p, "$(CIS_LAB[i])→$(CIS_LAB[j])\nn=$(length(x))", PDF_XLIM, PDF_YLIM;
             logx = false, logy = true)
end

function age_grid(s, kind)
    f = kind === :ccdf ? age_ccdf_panel : age_pdf_panel
    panels = [f(s, i, j) for i in 1:N_BIN for j in 1:N_BIN]
    p = plot(panels...; layout = (N_BIN, N_BIN), size = (1700, 1500),
             plot_title = "$(uppercase(string(kind))) — $(s)  (row = participant bin, col = contactee bin)",
             plot_titlefontsize = 11)
    savefig(p, "../res/6j_agepair_$(kind)_$(s).png")
    p
end
nothing

In [ ]:
age_grid(:home, :ccdf)

In [ ]:
age_grid(:nonhome, :ccdf)

In [ ]:
age_grid(:home, :pdf)

In [ ]:
age_grid(:nonhome, :pdf)

### §8 Takeaways

- **Home** age-pair cells are light-tailed: Weibull shape ≈ 2 (k > 1), so the
  duration-weighted degree concentrates around a typical value — consistent with
  small, repeated household-type contacts.
- **Non-home** cells are heavy-tailed: shape ≈ 0.6 (k < 1, sub-exponential), with a
  long right tail of high-weight participant-days — the Weibull captures the bulk but
  the extreme tail is heavier than a single Weibull implies.
- Every 7×7 cell (both settings) clears `MIN_N = 30`, so all 98 cells are fitted; the
  densest cells sit near the diagonal and in the middle-aged participant rows.
- ~65% of contacts carry an ambiguous contactee age interval, so the
  population-weighted bin assignment is doing real work; results are conditional on a
  single seeded draw (`MersenneTwister(1236)`). Re-running with another seed would
  perturb sparse off-diagonal cells most.
- The fits are *within-cell* MLEs; cell counts (`n` in each panel title and in
  `agetab`) should be read alongside the curves, since sparse cells give noisier fits.